# Superstore Sales Dataset Analysis
## Week 2: Exploratory Data Analysis & Strategic Business Questions

**Author:** Data Analysis Team  
**Project Phase:** Week 2 (Analysis Questions Phase)  
**Dataset:** `data/processed/superstore_clean.csv` (Output of Week 1 Preprocessing, 9,799 rows)  

---

### Business Objective
The goal of Week 2 is to answer high-impact business questions for C-suite executives, category managers, territory leads, and supply chain operators. We uncover revenue drivers, regional growth poles, customer loyalty dynamics, fulfillment bottlenecks, and product concentration patterns (Pareto 80/20).

## 1. Setup & Environment
Imports, project paths, display formatting, and Seaborn theme configuration.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'${x:,.2f}' if isinstance(x, (int, float)) and x > 10 else f'{x:,.2f}')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 130

ROOT = Path('..').resolve()
DATA_PATH = ROOT / 'data' / 'processed' / 'superstore_clean.csv'
FIG_DIR = ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])
print(f'Loaded cleaned dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head(3)

## 2. Executive Overview & Macro Financial KPIs
**Business Question:** What is our macro financial performance across the 4-year lifecycle?

We calculate top-line revenue, distinct orders count, total volume, average order value (AOV), active client accounts, and average delivery speed.

In [2]:
kpis = {
    'Total Revenue': f'${df["sales"].sum():,.2f}',
    'Total Unique Orders': f'{df["order_id"].nunique():,}',
    'Total Order Lines Sold': f'{len(df):,}',
    'Average Order Line Value': f'${df["sales"].mean():,.2f}',
    'Average Order Value (AOV)': f'${df["sales"].sum() / df["order_id"].nunique():,.2f}',
    'Average Fulfillment Days': f'{df["ship_days"].mean():.2f} days',
    'Active Customer Base': f'{df["customer_id"].nunique():,} clients',
    'Active Product SKUs': f'{df["product_id"].nunique():,} items'
}

kpi_df = pd.DataFrame(list(kpis.items()), columns=['Strategic Metric', 'Benchmark Value'])
kpi_df

## 3. Product Category & Sub-Category Contribution
**Business Question:** Which categories and sub-categories generate the largest revenue, and how does item volume compare to revenue share?

In [3]:
cat_summary = df.groupby('category').agg(
    total_sales=('sales', 'sum'),
    transaction_count=('row_id', 'count'),
    avg_ticket=('sales', 'mean')
).reset_index()
cat_summary['revenue_pct'] = (cat_summary['total_sales'] / df['sales'].sum()) * 100
cat_summary['volume_pct'] = (cat_summary['transaction_count'] / len(df)) * 100
cat_summary.sort_values('total_sales', ascending=False)

In [4]:
subcat = df.groupby(['category', 'sub_category'])['sales'].sum().reset_index()
subcat['pct_of_total'] = (subcat['sales'] / df['sales'].sum()) * 100
subcat = subcat.sort_values('sales', ascending=False)
subcat.head(10)

### Visualizing Category and Sub-Category Performance (Figure 05)

In [5]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [1, 2]})

cat_sales = df.groupby('category')['sales'].sum().sort_values(ascending=False)
bars = ax1.bar(cat_sales.index, cat_sales.values / 1000, color=['#2b5c8f', '#d95f02', '#7570b3'])
ax1.set_title('Total Revenue by Category ($K)', fontsize=13, weight='bold')
ax1.set_ylabel('Revenue ($ Thousands)')
for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, yval + 10, f'${yval:,.1f}K', ha='center', va='bottom', fontsize=10, weight='bold')

subcat_sales = df.groupby(['category', 'sub_category'])['sales'].sum().reset_index().sort_values('sales', ascending=True)
colors = {'Technology': '#2b5c8f', 'Furniture': '#d95f02', 'Office Supplies': '#7570b3'}
subcat_colors = [colors[c] for c in subcat_sales['category']]

ax2.barh(subcat_sales['sub_category'], subcat_sales['sales'] / 1000, color=subcat_colors)
ax2.set_title('Revenue by Sub-Category ($K)', fontsize=13, weight='bold')
ax2.set_xlabel('Revenue ($ Thousands)')
for i, v in enumerate(subcat_sales['sales'] / 1000):
    ax2.text(v + 3, i, f'${v:,.1f}K', va='center', fontsize=9)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=k) for k, c in colors.items()]
ax2.legend(handles=legend_elements, loc='lower right', title='Category')
plt.tight_layout()
plt.savefig(FIG_DIR / '05_sales_by_category_and_subcategory.png')
plt.show()

## 4. Geographic & Regional Territory Dominance
**Business Question:** Which territories and states drive the company's revenue, and where are market expansion opportunities needed?

In [6]:
reg = df.groupby('region').agg(
    total_sales=('sales', 'sum'),
    orders=('order_id', 'nunique'),
    avg_ship_days=('ship_days', 'mean')
).reset_index()
reg['revenue_share_pct'] = (reg['total_sales'] / df['sales'].sum()) * 100
reg.sort_values('total_sales', ascending=False)

In [7]:
state_sales = df.groupby(['state', 'region'])['sales'].sum().reset_index().sort_values('sales', ascending=False)
state_sales['cum_pct'] = (state_sales['sales'].cumsum() / df['sales'].sum()) * 100
print('=== TOP 10 STATES BY REVENUE ===')
state_sales.head(10)

### Visualizing Regional and State Performance (Figures 06 & 07)

In [8]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
reg_sales = df.groupby('region')['sales'].sum().sort_values(ascending=False)
ax1.pie(reg_sales, labels=reg_sales.index, autopct='%1.1f%%', startangle=140,
        colors=['#4575b4', '#74add1', '#abd9e9', '#fdae61'],
        textprops=dict(color='black', weight='bold'))
ax1.set_title('National Revenue Share by Region', fontsize=13, weight='bold')

reg_cat = df.pivot_table(index='region', columns='category', values='sales', aggfunc='sum')
reg_cat.plot(kind='bar', stacked=True, ax=ax2, colormap='viridis', edgecolor='white')
ax2.set_title('Regional Category Mix ($)', fontsize=13, weight='bold')
ax2.set_ylabel('Total Sales ($)')
ax2.set_xlabel('Region')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / '06_regional_sales_performance.png')
plt.show()

In [9]:
st_sales = df.groupby('state')['sales'].sum().sort_values(ascending=False)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
top10 = st_sales.head(10).sort_values(ascending=True)
ax1.barh(top10.index, top10.values / 1000, color='#1b7837')
ax1.set_title('Top 10 States by Revenue ($K)', fontsize=13, weight='bold')
ax1.set_xlabel('Revenue ($ Thousands)')
for i, v in enumerate(top10.values / 1000):
    ax1.text(v + 3, i, f'${v:,.1f}K', va='center', fontsize=9, weight='bold')

bottom10 = st_sales.tail(10).sort_values(ascending=True)
ax2.barh(bottom10.index, bottom10.values, color='#e41a1c')
ax2.set_title('Bottom 10 States by Revenue ($)', fontsize=13, weight='bold')
ax2.set_xlabel('Revenue ($)')
for i, v in enumerate(bottom10.values):
    ax2.text(v + 50, i, f'${v:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / '07_top_and_bottom_states.png')
plt.show()

## 5. Customer Segmentation & VIP Lifetime Value
**Business Question:** How do sales distribute among Consumer, Corporate, and Home Office segments, and who are our most loyal VIP accounts?

In [10]:
seg = df.groupby('segment').agg(
    total_revenue=('sales', 'sum'),
    unique_customers=('customer_id', 'nunique'),
    unique_orders=('order_id', 'nunique')
).reset_index()
seg['revenue_share_pct'] = (seg['total_revenue'] / df['sales'].sum()) * 100
seg['spend_per_customer'] = seg['total_revenue'] / seg['unique_customers']
seg.sort_values('total_revenue', ascending=False)

In [11]:
vip = df.groupby(['customer_id', 'customer_name', 'segment']).agg(
    lifetime_sales=('sales', 'sum'),
    orders_placed=('order_id', 'nunique'),
    items_bought=('row_id', 'count')
).reset_index().sort_values('lifetime_sales', ascending=False)
print('=== TOP 10 VIP ACCOUNTS ===')
vip.head(10)

### Customer Segmentation Visuals (Figure 08)

In [12]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.pie(seg['total_revenue'], labels=seg['segment'], autopct='%1.1f%%',
        colors=['#2b83ba', '#abdda4', '#fdae61'], startangle=90, explode=[0.05, 0, 0])
ax1.set_title('Revenue Contribution by Segment', fontsize=13, weight='bold')

x = np.arange(len(seg['segment']))
width = 0.35
ax2.bar(x - width/2, seg['unique_orders'], width, label='Unique Orders', color='#2b83ba')
ax2.bar(x + width/2, seg['unique_customers'], width, label='Unique Customers', color='#fdae61')
ax2.set_xticks(x)
ax2.set_xticklabels(seg['segment'], weight='bold')
ax2.set_ylabel('Count')
ax2.set_title('Orders & Customer Base by Segment', fontsize=13, weight='bold')
ax2.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '08_customer_segmentation_analysis.png')
plt.show()

## 6. Logistics & Fulfillment Performance
**Business Question:** What is the operational performance of each shipping tier, and are SLA fulfillment benchmarks met?

In [13]:
ship = df.groupby('ship_mode').agg(
    packages=('row_id', 'count'),
    avg_ship_days=('ship_days', 'mean'),
    median_ship_days=('ship_days', 'median'),
    min_days=('ship_days', 'min'),
    max_days=('ship_days', 'max')
).reindex(['Same Day', 'First Class', 'Second Class', 'Standard Class']).reset_index()
ship['package_share_pct'] = (ship['packages'] / len(df)) * 100
ship

In [14]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='ship_mode', y='ship_days', order=['Same Day', 'First Class', 'Second Class', 'Standard Class'],
            palette=['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3'], ax=ax1, hue='ship_mode', legend=False)
ax1.set_title('Distribution of Shipping Days by Mode', fontsize=13, weight='bold')
ax1.set_ylabel('Fulfillment Lag (Days)')
ax1.set_xlabel('Ship Mode')

ship_counts = df['ship_mode'].value_counts()[['Same Day', 'First Class', 'Second Class', 'Standard Class']]
bars = ax2.bar(ship_counts.index, ship_counts.values, color=['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3'])
ax2.set_title('Package Volume by Shipping Mode', fontsize=13, weight='bold')
ax2.set_ylabel('Total Line Shipments')
for bar in bars:
    yval = bar.get_height()
    pct = 100 * yval / len(df)
    ax2.text(bar.get_x() + bar.get_width()/2.0, yval + 50, f'{yval:,} ({pct:.1f}%)', ha='center', va='bottom', fontsize=9, weight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / '09_shipping_mode_lead_times.png')
plt.show()

## 7. Temporal Trends, YoY Growth & Seasonality Patterns
**Business Question:** Has revenue grown consistently year over year, and what seasonal quarters/months experience peak volume?

In [15]:
annual = df.groupby('order_year').agg(
    total_sales=('sales', 'sum'),
    order_count=('order_id', 'nunique'),
    items_count=('row_id', 'count')
).reset_index()
annual['yoy_growth_pct'] = annual['total_sales'].pct_change() * 100
annual

In [16]:
fig, ax1 = plt.subplots(figsize=(10, 5))
color = '#1f77b4'
ax1.set_xlabel('Order Year', fontsize=12, weight='bold')
ax1.set_ylabel('Total Revenue ($ Thousands)', color=color, fontsize=12, weight='bold')
bars = ax1.bar(annual['order_year'].astype(str), annual['total_sales'] / 1000, color=color, alpha=0.85, width=0.5)
ax1.tick_params(axis='y', labelcolor=color)
for bar in bars:
    y = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, y/2, f'${y:,.1f}K', ha='center', va='center', color='white', weight='bold', fontsize=11)

ax2 = ax1.twinx()
color = '#d62728'
ax2.set_ylabel('YoY Growth (%)', color=color, fontsize=12, weight='bold')
ax2.plot(annual['order_year'].astype(str), annual['yoy_growth_pct'], color=color, marker='o', linewidth=2.5, markersize=8)
ax2.tick_params(axis='y', labelcolor=color)
for i, txt in enumerate(annual['yoy_growth_pct']):
    if not np.isnan(txt):
        ax2.annotate(f'{txt:+.1f}%', (i, txt), textcoords='offset points', xytext=(0, 10), ha='center', weight='bold', color=color)

plt.title('Annual Sales Performance & Year-Over-Year Growth Trajectory', fontsize=13, weight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '10_annual_sales_and_yoy_growth.png')
plt.show()

In [17]:
monthly_pivot = df.pivot_table(index='order_month', columns='order_year', values='sales', aggfunc='sum')
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_pivot.index = months

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
for col in monthly_pivot.columns:
    ax1.plot(monthly_pivot.index, monthly_pivot[col] / 1000, marker='o', label=f'Year {col}', linewidth=2)
ax1.set_title('Monthly Sales Trends by Year ($K)', fontsize=13, weight='bold')
ax1.set_ylabel('Sales ($ Thousands)')
ax1.set_xlabel('Month')
ax1.legend()

sns.heatmap(monthly_pivot / 1000, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax2, cbar_kws={'label': 'Sales ($K)'})
ax2.set_title('Monthly Sales Seasonality Heatmap ($K)', fontsize=13, weight='bold')
ax2.set_ylabel('Month')
ax2.set_xlabel('Year')
plt.tight_layout()
plt.savefig(FIG_DIR / '11_monthly_seasonality_patterns.png')
plt.show()

## 8. Revenue Concentration & Product Pareto Analysis (80/20 Rule)
**Business Question:** Does the Pareto 80/20 rule hold for our product catalog, and what is the impact of transaction outliers?

In [18]:
outlier_summary = df.groupby('is_sales_outlier').agg(
    transactions=('row_id', 'count'),
    total_revenue=('sales', 'sum'),
    avg_ticket=('sales', 'mean')
).reset_index()
outlier_summary['volume_pct'] = (outlier_summary['transactions'] / len(df)) * 100
outlier_summary['revenue_pct'] = (outlier_summary['total_revenue'] / df['sales'].sum()) * 100
outlier_summary

In [19]:
prod_sales = df.groupby('product_id')['sales'].sum().sort_values(ascending=False).reset_index()
prod_sales['cum_sales'] = prod_sales['sales'].cumsum()
prod_sales['cum_sales_pct'] = 100.0 * prod_sales['cum_sales'] / prod_sales['sales'].sum()
prod_sales['cum_prod_pct'] = 100.0 * (prod_sales.index + 1) / len(prod_sales)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(prod_sales['cum_prod_pct'], prod_sales['cum_sales_pct'], color='#08519c', linewidth=2.5, label='Cumulative Sales %')
ax.plot([0, 100], [0, 100], color='gray', linestyle='--', label='Equal Distribution Line')

idx_80 = (prod_sales['cum_sales_pct'] >= 80).idxmax()
prod_pct_at_80 = prod_sales.loc[idx_80, 'cum_prod_pct']

ax.axhline(80, color='red', linestyle=':', alpha=0.8)
ax.axvline(prod_pct_at_80, color='red', linestyle=':', alpha=0.8)
ax.scatter([prod_pct_at_80], [80], color='red', s=80, zorder=5)
ax.annotate(f'80% of Sales generated by top {prod_pct_at_80:.1f}% Products',
            xy=(prod_pct_at_80, 80), xytext=(prod_pct_at_80 + 10, 70),
            arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6),
            fontsize=11, weight='bold', color='#a50f15')

ax.set_title('Pareto Analysis: Cumulative Product Share vs. Cumulative Revenue', fontsize=13, weight='bold')
ax.set_xlabel('Cumulative % of Product Catalog')
ax.set_ylabel('Cumulative % of Revenue')
ax.set_xlim(0, 100)
ax.set_ylim(0, 105)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(FIG_DIR / '12_product_pareto_curve.png')
plt.show()

## 9. Strategic Executive Insights & Decisions Summary

1. **Catalog Rebalancing:** 22.4% of products drive 80% of sales. Secure priority vendor terms and prevent stockouts for top subcategories (Phones, Chairs, Storage).
2. **Regional Fulfillment Hubs:** California and New York alone represent 33.3% of national revenue. Establish localized inventory hubs in these states to compress Standard Class delivery times from 5 days to 2–3 days.
3. **VIP Loyalty Lock-in:** The top 10 customers drive significant volume with high retention. Formalize dedicated corporate account management to protect account health.
4. **Q4 Ramp-Up:** Q4 accounts for ~34% of annual revenue, peaking in November and December. Working capital and inventory replenishment must be prioritized in August/September to capture full holiday demand.